In [1]:
cd ..

/home/serafeim/Desktop/bet


In [2]:
from src.utils.spark_session import get_spark
from pyspark.sql import functions as F
from pyspark.sql.window import Window
import pandas as pd
from pyspark.sql.functions import pandas_udf
import src.utils.config as config
#from src.ingestion.churn_label_generator import generate_churn_labels


In [3]:
spark = get_spark()
spark.catalog.clearCache()
config_ = config.DataGenConfig()
players_bronze = spark.read.parquet("./data/bronze/players")
sessions_bronze = spark.read.parquet("./data/bronze/sessions")
transactions_bronze = spark.read.parquet("./data/bronze/transactions")


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/02 19:01:29 WARN Utils: Your hostname, serafeim-virtual-machine, resolves to a loopback address: 127.0.1.1; using 10.100.2.34 instead (on interface ens192)
26/03/02 19:01:29 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/02 19:01:41 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/03/02 19:02:12 WARN SharedInMemoryCache: Evicting cached table partition metadata from memory due to size constraints (spark.sql.hive.filesourcePartitionFileCacheSize = 262144000 bytes). This may impact query planning performance.


In [6]:
sessions_bronze.count()

84776

In [7]:
sessions_bronze.columns

['session_id',
 'player_id',
 'game_id',
 'session_duration_sec',
 'bet_count',
 'total_bet_amount',
 'total_win_amount',
 'session_date']

In [9]:
sessions_bronze = sessions_bronze.dropDuplicates(['player_id', 'session_date'])
sessions_bronze.show()

+--------------------+---------+-------+--------------------+---------+----------------+----------------+-------------------+
|          session_id|player_id|game_id|session_duration_sec|bet_count|total_bet_amount|total_win_amount|       session_date|
+--------------------+---------+-------+--------------------+---------+----------------+----------------+-------------------+
|598acccd-d537-4f1...|       P0|    G43|                1229|       11|           38.44|           24.53|2024-01-01 09:45:00|
|6f2e370d-9f10-464...|       P0|   G125|                3283|        8|            45.3|            93.3|2024-01-07 05:51:36|
|447c5989-f5ff-4de...|       P0|    G55|                1224|       12|           58.92|           19.48|2024-01-09 19:08:25|
|ea5f2db4-6c87-467...|       P0|   G187|                 762|        0|           46.26|           11.64|2024-01-20 23:40:21|
|5111a10b-2234-4b2...|       P0|    G21|                 755|       14|            41.2|          117.35|2024-02-04 01

In [10]:
sessions_bronze.count()

84776